In [1]:
import sys
import os
import json
import asyncio
from pathlib import Path

# 添加项目路径
sys.path.append(os.path.dirname(os.path.abspath('.')))

from src.manager.mcp_client_manager import MCPClientManager
from src.gen.validate import ValidateGen
from src.utils.utils import load_metadata

print("✓ 导入成功")


✓ 导入成功


In [2]:
# 初始化 MCPClientManager
print("初始化 MCPClientManager...")
client_manager = MCPClientManager()
client_manager.init_config("configs/mcp_server.json", overwrite=True)
print("✓ MCPClientManager 初始化完成")

# 检查 GoogleMaps 是否已注册
googlemaps_tools = [tool for tool in client_manager.tool_schemas if tool['function']['name'].startswith('GoogleMaps-')]
print(f"\n找到 {len(googlemaps_tools)} 个 GoogleMaps 工具:")
for tool in googlemaps_tools:
    print(f"  - {tool['function']['name']}")


初始化 MCPClientManager...
✓ MCPClientManager 初始化完成

找到 9 个 GoogleMaps 工具:
  - GoogleMaps-load_scenario
  - GoogleMaps-save_scenario
  - GoogleMaps-maps_geocode
  - GoogleMaps-maps_reverse_geocode
  - GoogleMaps-maps_search_places
  - GoogleMaps-maps_place_details
  - GoogleMaps-maps_distance_matrix
  - GoogleMaps-maps_elevation
  - GoogleMaps-maps_directions


In [3]:
from agents import set_tracing_disabled
set_tracing_disabled(disabled=True)
# 读取 GoogleMaps metadata 和代码
metadata_path = "envs/metadata/GoogleMaps-metadata.json"
tool_code_path = "envs/tools/GoogleMaps.py"

# 加载 metadata
metadata = load_metadata(metadata_path)
mcp_server_name = metadata['class_name']
print(f"MCP Server Name: {mcp_server_name}")
print(f"Description: {metadata['description']}")
print(f"Tools: {len(metadata['tools'])} 个工具")

# 读取工具代码
tool_code = Path(tool_code_path).read_text(encoding='utf-8')
print(f"\n✓ 代码长度: {len(tool_code)} 字符")

# 初始化 ValidateGen
print("\n初始化 ValidateGen...")
validator = ValidateGen(client_manager=client_manager, log_interactions=True)
print("✓ ValidateGen 初始化完成")


MCP Server Name: GoogleMaps
Description: Leverage the power of Google Maps to enhance your applications with location services, directions, and place details. Integrate seamlessly with your LLMs to provide real-time geographic data and insights. Start building location-aware applications effortlessly with this MCP server.
Tools: 7 个工具

✓ 代码长度: 17890 字符

初始化 ValidateGen...
✓ ValidateGen 初始化完成


In [4]:
# 测试 1: 验证 scenario consistency
print("=" * 60)
print("测试 1: 验证 Scenario Consistency")
print("=" * 60)

async def test_scenario_consistency():
    result = await validator.validate_scenario_consistency(
        mcp_server_name=mcp_server_name,
        tool_code=tool_code,
        tool_file_path=tool_code_path
    )
    
    print(f"\n验证结果:")
    print(f"  - is_valid: {result['is_valid']}")
    print(f"  - consistency_check: {result.get('consistency_check', 'N/A')}")
    
    if result.get('errors'):
        print(f"  - 错误: {result['errors']}")
    
    if result.get('load_result'):
        print(f"\n  - load_result: {result['load_result'][:200]}...")
    
    if result.get('save_result'):
        print(f"\n  - save_result 类型: {type(result['save_result'])}")
        if isinstance(result['save_result'], dict):
            print(f"  - save_result keys: {list(result['save_result'].keys())}")
    
    return result

# 运行测试
scenario_result = await test_scenario_consistency()


13:18:58 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:18:58,194 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


测试 1: 验证 Scenario Consistency


13:19:26 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:19:26,523 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


Load scenario succeeded without checking: Successfully loaded from scenario!


13:19:30 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:19:30,572 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


GoogleMaps-save_scenario execute: {
  "geocoded_addresses": {
    "1600 Amphitheatre Parkway, Mountain View, CA": {
      "latitude": 37.422,
      "longitude": -122.084,
      "formatted_address": "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA"
    }
  },
  "reverse_geocoded_locations": {
    "37.422,-122.084": "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA"
  },
  "places": {
    "ChIJN1t_tDeuEmsRUsoyG83frY4": {
      "name": "Googleplex",
      "address": "1600 Amphitheatre Parkway, Mountain View, CA",
      "place_id": "ChIJN1t_tDeuEmsRUsoyG83frY4"
    }
  },
  "place_details": {
    "ChIJN1t_tDeuEmsRUsoyG83frY4": {
      "name": "Googleplex",
      "address": "1600 Amphitheatre Parkway, Mountain View, CA 94043, USA",
      "phone_number": "+1 650-253-0000",
      "website": "https://www.google.com"
    }
  },
  "distance_matrix_results": {
    "Mountain View, CA|San Francisco, CA": [
      {
        "origin": "Mountain View, CA",
        "destination": "San Fra

2025-11-17 13:20:12,288 - src.gen.validate - INFO - Tool: scenario_consistency | Request ID: dcf2a57392b1490e9f040abb09b925ac
2025-11-17 13:20:12,291 - src.gen.validate - INFO - [
  {
    "user": "# Scenario Consistency Validation Request\n\n## MCP Server Name\nGoogleMaps\n\n## Tool Implementation Code\n```python\nfrom pydantic import BaseModel, Field\nfrom typing import Dict, List, Optional, Union, Any\n\nclass GeocodeResult(BaseModel):\n    \"\"\"Geocoding result data structure\"\"\"\n    latitude: float = Field(description=\"The latitude coordinate of the address\")\n    longitude: float = Field(description=\"The longitude coordinate of the address\")\n    formatted_address: str = Field(description=\"The complete formatted address\")\n\nclass Place(BaseModel):\n    \"\"\"Place data structure\"\"\"\n    name: str = Field(description=\"Name of the place\")\n    address: str = Field(description=\"Address of the place\")\n    place_id: str = Field(description=\"Unique identifier for the

Client GoogleMaps-dcf2a57392b1490e9f040abb09b925ac closed and removed

验证结果:
  - is_valid: True
  - consistency_check: True

  - load_result: Successfully loaded from scenario!...

  - save_result 类型: <class 'dict'>
  - save_result keys: ['geocoded_addresses', 'reverse_geocoded_locations', 'places', 'place_details', 'distance_matrix_results', 'elevation_data', 'directions']


In [5]:
# 测试 2: 验证单个工具执行 (maps_geocode)
print("\n" + "=" * 60)
print("测试 2: 验证工具执行 - maps_geocode")
print("=" * 60)

# 找到 maps_geocode 工具的 metadata
maps_geocode_tool = None
for tool in metadata['tools']:
    if tool['name'] == 'maps_geocode':
        maps_geocode_tool = tool
        break

if maps_geocode_tool:
    print(f"\n工具信息:")
    print(f"  - 名称: {maps_geocode_tool['name']}")
    print(f"  - 描述: {maps_geocode_tool['description']}")
    print(f"  - 输入参数: {list(maps_geocode_tool['input_schema']['properties'].keys())}")
    
    async def test_tool_execution():
        result = await validator.validate_tool_execution(
            tool=maps_geocode_tool,
            mcp_server_name=mcp_server_name,
            tool_code=tool_code,
            request_id="test_request_001"
        )
        
        print(f"\n验证结果:")
        print(f"  - is_valid: {result['is_valid']}")
        
        summary = result.get('summary', {})
        print(f"\n测试摘要:")
        print(f"  - 总测试用例数: {summary.get('total_test_cases', 0)}")
        print(f"  - 通过数: {summary.get('passed_count', 0)}")
        print(f"  - 失败数: {summary.get('failed_count', 0)}")
        print(f"  - 整体通过: {summary.get('overall_passed', False)}")
        
        if summary.get('errors'):
            print(f"\n错误:")
            for error in summary['errors']:
                print(f"  - {error}")
        
        test_cases = result.get('test_cases', [])
        print(f"\n测试用例详情 ({len(test_cases)} 个):")
        for i, test_case in enumerate(test_cases, 1):
            print(f"\n  测试用例 {i}:")
            print(f"    - test_case_id: {test_case.get('test_case_id', 'N/A')}")
            print(f"    - client_id: {test_case.get('client_id', 'N/A')}")
            print(f"    - passed: {test_case.get('passed', False)}")
            if test_case.get('evaluation'):
                eval_text = str(test_case['evaluation'])[:150]
                print(f"    - evaluation: {eval_text}...")
        
        return result
    
    # 运行测试
    tool_result = await test_tool_execution()
else:
    print("❌ 未找到 maps_geocode 工具")


13:22:00 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:22:00,272 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek



测试 2: 验证工具执行 - maps_geocode

工具信息:
  - 名称: maps_geocode
  - 描述: Convert address into geographic coordinates
  - 输入参数: ['address']


13:22:19 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:22:19,930 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


Load scenario succeeded without checking: Successfully loaded from scenario!


13:22:23 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:22:23,168 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


...

GoogleMaps-maps_geocode execute: {
  "error": "Address not found in geocoded data"
}


13:23:13 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:23:13,501 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


GoogleMaps-save_scenario execute: {
  "geocoded_addresses": {
    "1600 Amphitheatre Parkway, Mountain View, CA": {
      "latitude": 37.4220656,
      "longitude": -122.0840897,
      "formatted_address": "1600 Amphitheatre Pkwy, Mountain View, CA 94043, USA"
    },
    "1 Infinite Loop, Cupertino, CA": {
      "latitude": 37.33182,
      "longitude": -122.03118,
      "formatted_address": "1 Infinite Loop, Cupertino, CA 95014, USA"
    },
    "350 5th Ave, New York, NY": {
      "latitude": 40.748817,
      "longitude": -73.985428,
      "formatted_address": "350 5th Ave, New York, NY 10118, USA"
    }
  },
  "reverse_geocoded_locations": {
    "37.4220656,-122.0840897": "1600 Amphitheatre Pkwy, Mountain View, CA 94043, USA"
  },
  "places": {
    "ChIJiT3W8dq1j4ARy7t1C4g8o6w": {
      "name": "Googleplex",
      "address": "1600 Amphitheatre Pkwy, Mountain View, CA 94043, USA",
      "place_id": "ChIJiT3W8dq1j4ARy7t1C4g8o6w"
    }
  },
  "place_details": {
    "ChIJiT3W8dq1j4ARy7t1C

2025-11-17 13:24:25,450 - src.gen.validate - INFO - Tool: maps_geocode | Request ID: test_request_001
2025-11-17 13:24:25,453 - src.gen.validate - INFO - [
  {
    "user": "# MCP Tool Test Request\n\n## MCP Server Name\nGoogleMaps\n\n## Tool Information\nTool Name: maps_geocode\nDescription: Convert address into geographic coordinates\n\n## Tool Input Schema\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"address\": {\n      \"type\": \"string\",\n      \"description\": \"The address to geocode\"\n    }\n  },\n  \"required\": [\n    \"address\"\n  ]\n}\n\n## Tool Output Schema\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"latitude\": {\n      \"type\": \"number\",\n      \"description\": \"The latitude coordinate of the address\"\n    },\n    \"longitude\": {\n      \"type\": \"number\",\n      \"description\": \"The longitude coordinate of the address\"\n    },\n    \"formatted_address\": {\n      \"type\": \"string\",\n      \"description\": \"The complete formatted

Client GoogleMaps-test_request_001_test_test001 closed and removed
Client GoogleMaps-test_request_001_test_test002 closed and removed
Client GoogleMaps-test_request_001_test_test003 closed and removed

验证结果:
  - is_valid: True

测试摘要:
  - 总测试用例数: 3
  - 通过数: 3
  - 失败数: 0
  - 整体通过: True

测试用例详情 (3 个):

  测试用例 1:
    - test_case_id: test001
    - client_id: GoogleMaps-test_request_001_test_test001
    - passed: True
    - evaluation: Test case passed: Tool successfully returned geocoding data for Google headquarters. Output matches expected structure with correct latitude, longitud...

  测试用例 2:
    - test_case_id: test002
    - client_id: GoogleMaps-test_request_001_test_test002
    - passed: True
    - evaluation: Test case passed: Tool successfully returned geocoding data for Apple headquarters. Output matches expected structure with correct latitude, longitude...

  测试用例 3:
    - test_case_id: test003
    - client_id: GoogleMaps-test_request_001_test_test003
    - passed: True
    - ev

In [6]:
# 测试 3: 验证另一个工具 (maps_search_places)
print("\n" + "=" * 60)
print("测试 3: 验证工具执行 - maps_search_places")
print("=" * 60)

# 找到 maps_search_places 工具的 metadata
maps_search_tool = None
for tool in metadata['tools']:
    if tool['name'] == 'maps_search_places':
        maps_search_tool = tool
        break

if maps_search_tool:
    print(f"\n工具信息:")
    print(f"  - 名称: {maps_search_tool['name']}")
    print(f"  - 描述: {maps_search_tool['description']}")
    print(f"  - 输入参数: {list(maps_search_tool['input_schema']['properties'].keys())}")
    
    async def test_search_tool():
        result = await validator.validate_tool_execution(
            tool=maps_search_tool,
            mcp_server_name=mcp_server_name,
            tool_code=tool_code,
            request_id="test_request_002"
        )
        
        print(f"\n验证结果:")
        print(f"  - is_valid: {result['is_valid']}")
        
        summary = result.get('summary', {})
        print(f"\n测试摘要:")
        print(f"  - 总测试用例数: {summary.get('total_test_cases', 0)}")
        print(f"  - 通过数: {summary.get('passed_count', 0)}")
        print(f"  - 失败数: {summary.get('failed_count', 0)}")
        print(f"  - 整体通过: {summary.get('overall_passed', False)}")
        
        return result
    
    # 运行测试
    search_result = await test_search_tool()
else:
    print("❌ 未找到 maps_search_places 工具")


13:26:41 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:26:41,994 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek



测试 3: 验证工具执行 - maps_search_places

工具信息:
  - 名称: maps_search_places
  - 描述: Search for places using Google Places API
  - 输入参数: ['query', 'radius', 'location']


13:26:55 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:26:55,622 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


Load scenario succeeded without checking: Successfully loaded from scenario!


13:26:59 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:26:59,469 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


...

GoogleMaps-maps_search_places execute: {
  "places": []
}


13:27:31 - LiteLLM:INFO: utils.py:3422 - 
LiteLLM completion() model= deepseek-chat; provider = deepseek
2025-11-17 13:27:31,841 - LiteLLM - INFO - 
LiteLLM completion() model= deepseek-chat; provider = deepseek


GoogleMaps-save_scenario execute: {
  "geocoded_addresses": {},
  "reverse_geocoded_locations": {},
  "places": {
    "place_009": {
      "name": "Tech Company HQ",
      "address": "444 Silicon Valley, San Francisco, CA",
      "place_id": "place_009"
    },
    "place_010": {
      "name": "Golden Gate Bridge",
      "address": "Golden Gate Bridge, San Francisco, CA",
      "place_id": "place_010"
    }
  },
  "place_details": {},
  "distance_matrix_results": {},
  "elevation_data": {},
  "directions": {}
}


2025-11-17 13:28:21,892 - src.gen.validate - INFO - Tool: maps_search_places | Request ID: test_request_002
2025-11-17 13:28:21,895 - src.gen.validate - INFO - [
  {
    "user": "# MCP Tool Test Request\n\n## MCP Server Name\nGoogleMaps\n\n## Tool Information\nTool Name: maps_search_places\nDescription: Search for places using Google Places API\n\n## Tool Input Schema\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"query\": {\n      \"type\": \"string\",\n      \"description\": \"The search query for places\"\n    },\n    \"radius\": {\n      \"type\": \"string\",\n      \"description\": \"Search radius in meters\"\n    },\n    \"location\": {\n      \"type\": \"string\",\n      \"description\": \"Location coordinates for search\"\n    }\n  },\n  \"required\": [\n    \"query\"\n  ]\n}\n\n## Tool Output Schema\n{\n  \"type\": \"object\",\n  \"properties\": {\n    \"places\": {\n      \"type\": \"array\",\n      \"items\": {\n        \"type\": \"object\",\n        \"properties\":

Client GoogleMaps-test_request_002_test_test001 closed and removed
Client GoogleMaps-test_request_002_test_test002 closed and removed
Client GoogleMaps-test_request_002_test_test003 closed and removed

验证结果:
  - is_valid: True

测试摘要:
  - 总测试用例数: 3
  - 通过数: 3
  - 失败数: 0
  - 整体通过: True


In [7]:
# 总结
print("\n" + "=" * 60)
print("测试总结")
print("=" * 60)

print(f"\n1. Scenario Consistency 验证:")
print(f"   - 状态: {'✓ 通过' if scenario_result.get('is_valid') else '✗ 失败'}")

if 'tool_result' in locals():
    print(f"\n2. maps_geocode 工具验证:")
    print(f"   - 状态: {'✓ 通过' if tool_result.get('is_valid') else '✗ 失败'}")
    print(f"   - 测试用例: {tool_result.get('summary', {}).get('total_test_cases', 0)} 个")

if 'search_result' in locals():
    print(f"\n3. maps_search_places 工具验证:")
    print(f"   - 状态: {'✓ 通过' if search_result.get('is_valid') else '✗ 失败'}")
    print(f"   - 测试用例: {search_result.get('summary', {}).get('total_test_cases', 0)} 个")

print("\n✓ 所有测试完成！")



测试总结

1. Scenario Consistency 验证:
   - 状态: ✓ 通过

2. maps_geocode 工具验证:
   - 状态: ✓ 通过
   - 测试用例: 3 个

3. maps_search_places 工具验证:
   - 状态: ✓ 通过
   - 测试用例: 3 个

✓ 所有测试完成！
